# Workshop: Lakeflow Declarative Pipeline

**Scenario:** Build a complete RetailHub medallion pipeline and then explain why each layer exists. First you will wire the pipeline in the Databricks UI, then you will write the Lakeflow SQL declarations and verify the result.

**Learning objective:** Build, run, and verify a Lakeflow Spark Declarative Pipeline end-to-end — declarations, ST vs MV, post-run verification, event-log analytics, and the data-quality quarantine.

**Expected duration:** ~45 minutes in class (Section 1: ~20', Tasks 1–5: ~25'; Tasks 6–7 are stretch)

This lab mirrors the full production flow:
1. Build and run the pipeline in the UI
2. Configure pipeline variables and source assets
3. Declare Bronze, Silver, and Gold layers in SQL
4. Classify STREAMING TABLE vs MATERIALIZED VIEW from catalog metadata
5. Verify row counts, query the Event Log, and audit the expectations quarantine

## Learning Focus

- Build a Lakeflow pipeline end-to-end in the UI
- Use `STREAMING TABLE` for Bronze ingestion
- Apply `EXPECT ... ON VIOLATION DROP ROW` for Silver quality gates
- Use `MATERIALIZED VIEW` for Gold aggregations
- Read the Event Log (`event_log(TABLE(...))`) to verify pipeline behavior
- Quantify how many dirty rows the expectations quarantined out of Silver

## Setup

In [ ]:
%run ../../setup/00_setup

In [ ]:
# Lakeflow pipeline target schema (matches Step 2 config)
user_name = CATALOG.replace(f"{CATALOG_PREFIX}_", "")
user_schema = f"{user_name}_lakeflow"
print(f"Pipeline target schema: {user_schema}")

## Section 1: Workshop — Building the Pipeline

In this hands-on workshop, we create a complete Lakeflow pipeline from SQL files — uploading source code, configuring the pipeline in the Databricks UI, running it, and validating the results.



### SQL Files Overview

The pipeline source code is organized by medallion layer — each SQL file declares one table or view.

![../../../assets/images/training_2026/day3/57bea06e969c45dab4de8bea9ec980b1.webp](../../../assets/images/training_2026/day3/57bea06e969c45dab4de8bea9ec980b1.webp)

### Step 1: Upload SQL Files

**Option A: Via UI** — Workspace → Users → create `lakeflow_pipeline` folder → upload SQL files

**Option B: Via Git Folders** — Git Folders → Add Git Folder → clone training repository

### Step 2: Create Pipeline

1. **Jobs & Pipelines** → **Create** → **ETL pipeline**
2. **Catalog:** `retailhub_<your_name>`
3. **Pipeline Name:** `lakeflow_pipeline_<your_name>`
4. **Target Schema:** `<your_name>_lakeflow`
5. **Source Code:** Add existing assets → choose pipeline root folder and source code folder

<img src="../../../assets/images/fab4fef8e72d4d5786ba818e6c2f73c5.png" width="800">

<img src="../../../assets/images/a4e21cb35d3b45f2ba184877139cdc48.png" width="800">

<img src="../../../assets/images/4e75e96544f142508c3bd8b0b4dbf446.png" width="800">

![../../../assets/images/training_2026/day3/dc9c3eea154e4cb29aee62e6edd5bcca.webp](../../../assets/images/training_2026/day3/dc9c3eea154e4cb29aee62e6edd5bcca.webp)

### Step 3: Configure Variables

**Configuration** → **Add configuration**:

| Key | Value |
|-----|-------|
| `customer_path` | `/Volumes/<your catalog>/default/datasets/customers` |
| `order_path` | `/Volumes/<your catalog>/default/datasets/orders` |
| `product_path` | `/Volumes/<your catalog>/default/datasets/products/products.parquet/` |

Open settings and go to Pipeline Configuration 

<img src="../../../assets/images/b2fa841a52004939ac2679f9edef8dc3.png" width="800">

Add configuration : 

<img src="../../../assets/images/dc31d5dec7444c1d959b8e1f220c10ce.png" width="800">

<img src="../../../assets/images/6f16ad4f7fd941ebbb4fb286dbb8fbfd.png" width="800">

You should also see a DAG diagram built based on Spark Declarative Pipelines definition

![../../../assets/images/training_2026/day3/05f00c9fedb54b0b81ec65bf182a92af.webp](../../../assets/images/training_2026/day3/05f00c9fedb54b0b81ec65bf182a92af.webp)

### Step 4: Run the Pipeline

Start the pipeline and test incremental processing by adding new data files.

![../../../assets/images/training_2026/day3/8d06de8a2a674cc1bc119b5d91b2d1ce.webp](../../../assets/images/training_2026/day3/8d06de8a2a674cc1bc119b5d91b2d1ce.webp)

1. Add new file to folder orders/stream/ from /Volumes/.../default/datasets/demo/ingestion/orders/stream/
2. Run pipeline again
3. Check Event Log - should process only new files

### Step 5: Verify Results

In [ ]:
# Check fact_sales with joins to dimensions
display(spark.sql(f"""
    SELECT 
        f.order_id,
        c.first_name || ' ' || c.last_name AS customer_name,
        p.product_name,
        d.date,
        f.quantity,
        f.net_amount
    FROM {CATALOG}.{user_schema}.fact_sales f
    LEFT JOIN {CATALOG}.{user_schema}.dim_customer c ON f.customer_key = c.customer_key
    LEFT JOIN {CATALOG}.{user_schema}.dim_product p ON f.product_key = p.product_key
    LEFT JOIN {CATALOG}.{user_schema}.dim_date d ON f.order_date_key = d.date_key
    LIMIT 10
"""))

In [ ]:
# Find customers with change history
display(spark.sql(f"""
    SELECT 
        customer_id, first_name, city,
        __START_AT, __END_AT,
        CASE WHEN __END_AT IS NULL THEN 'Current' ELSE 'Historical' END AS status
    FROM {CATALOG}.{user_schema}.silver_customers
    WHERE customer_id IN (
        SELECT customer_id 
        FROM {CATALOG}.{user_schema}.silver_customers 
        GROUP BY customer_id HAVING COUNT(*) > 1
    )
    ORDER BY customer_id, __START_AT
"""))

### Monitoring and Troubleshooting

Common issues encountered when running Lakeflow pipelines and how to resolve them.

| Issue | Cause | Solution |
|---------|-----------|-------------|
| Pipeline hangs | Cluster too small | Increase min workers |
| Missing data | Constraint DROP ROW | Check Data Quality tab |
| Schema mismatch | Schema change | Full refresh |

## Section 2: Practice — Lakeflow SQL Declarations

Write and verify Lakeflow SQL syntax for each medallion layer.

## Task 1: Write Bronze Declaration

Complete the SQL below to create a Bronze streaming table from JSON files.

**What you need to do:** Replace every `____` placeholder:
1. `CREATE OR REFRESH ____ TABLE` — the keyword that makes the table **incremental**
2. `FROM STREAM ____(...)` — the Lakeflow **file-ingestion function**
3. Inside the function — the source path (`/Volumes/{CATALOG}/default/datasets/orders/stream/`) and `format => 'json'`

**This SQL would go in a pipeline SQL file.** Here we practice the syntax.

**Guidance — Task 01**

The goal is to declare a **Bronze streaming table** that ingests raw JSON files using Lakeflow's `read_files` function.

**STREAMING TABLE vs regular table**
A `STREAMING TABLE` processes data incrementally — it tracks what has been read and only picks up new files on subsequent pipeline runs. This is the standard pattern for Bronze ingestion. The `STREAM` keyword before `read_files(...)` enables incremental file processing: Lakeflow creates a checkpoint and on re-run processes only new files, not the entire directory.

**Why `read_files` instead of `spark.read.format(...)`?**
`read_files()` is the Lakeflow-native function for file ingestion. It integrates with Auto Loader, handles schema evolution, and supports all common formats (`json`, `csv`, `parquet`). Unlike `spark.read`, it supports both batch (`read_files`) and streaming (`STREAM read_files`) modes within a Lakeflow pipeline.

In [ ]:
# Practice: write the Bronze SQL declaration syntax
# (This won't execute outside of a Lakeflow pipeline — the validation checks the syntax)

# TODO: Replace every ____ below
#   1. keyword before TABLE that makes the table incremental
#   2. Lakeflow function that reads files from a path
#   3. source path + format => 'json'
bronze_sql = f"""
CREATE OR REFRESH ____ TABLE bronze_orders
AS SELECT *
FROM STREAM ____(
    ____
);
"""

print("Bronze SQL declaration:")
print(bronze_sql)


In [ ]:
# -- Validation --
assert "____" not in bronze_sql, "Replace every ____ placeholder in bronze_sql"
_sql = " ".join(bronze_sql.upper().split())          # normalise whitespace
assert "CREATE OR REFRESH STREAMING TABLE BRONZE_ORDERS" in _sql, "Should declare a STREAMING TABLE bronze_orders"
assert "STREAM READ_FILES(" in _sql, "Should read incrementally with STREAM read_files(...)"
assert "FORMAT =>" in _sql and "/VOLUMES/" in _sql, "Pass the Volume path and format => 'json' to read_files()"
print("Task 1 OK: Bronze declaration syntax correct")


## Task 2: Write Silver Declaration with Expectations

Complete the Silver layer with data quality constraints.

**What you need to do:** Complete `ON VIOLATION ...` — use keyword: `DROP ROW` for both constraints.

**Guidance — Task 02**

The goal is to add **data quality expectations** to the Silver layer — the first line of defense against bad data.

**How expectations work**
Expectations are `CONSTRAINT` declarations in the `CREATE` statement. Each constraint has a name, a boolean expression, and a violation action.

| Action | Behavior | When to use |
|--------|----------|-------------|
| `ON VIOLATION DROP ROW` | Row removed silently; violation logged | Business rule violations (null IDs, negative quantities) |
| `ON VIOLATION FAIL UPDATE` | Entire pipeline run fails | Critical integrity constraints — no data is better than bad data |
| *(no action)* | Row kept; violation count logged only | Monitoring without blocking |

**Exam Tip:** `DROP ROW` is the most commonly tested action. Know that it silently discards the row and logs the violation count in the Event Log — it does NOT fail the pipeline.

In [ ]:
# TODO: Complete Silver SQL with data quality expectations
# ON VIOLATION options:
#   DROP ROW  — silently drop rows that violate the constraint
#   FAIL UPDATE — fail the entire pipeline update if any row violates
# Use DROP ROW for recoverable data issues, FAIL for critical integrity constraints

silver_sql = """
CREATE OR REFRESH STREAMING TABLE silver_orders (
    CONSTRAINT valid_id EXPECT (order_id IS NOT NULL) ON VIOLATION ...,
    CONSTRAINT positive_amount EXPECT (total_price > 0) ON VIOLATION ...
)
AS SELECT 
    order_id,
    customer_id,
    product_id,
    CAST(quantity AS INT) AS quantity,
    CAST(total_price AS DOUBLE) AS total_price,
    CAST(order_date AS DATE) AS order_date,
    payment_method,
    store_id,
    current_timestamp() AS processed_at
FROM STREAM(bronze_orders);
"""

print("Silver SQL declaration:")
print(silver_sql)

In [ ]:
# -- Validation --
assert "CONSTRAINT" in silver_sql.upper(), "Should have CONSTRAINT declarations"
assert "DROP ROW" in silver_sql.upper(), "Should use ON VIOLATION DROP ROW"
assert "bronze" in silver_sql.lower(), "Should reference bronze_orders"
print("Task 2 OK: Silver declaration with expectations correct")

## Task 3: Write Gold Declaration

Create a Materialized View for daily revenue summary.

**What you need to do:** Complete `CREATE OR REFRESH ...` — use: `MATERIALIZED VIEW`

**Guidance — Task 03**

The goal is to write a **Gold Materialized View** — the final aggregated layer optimized for analytics.

**MATERIALIZED VIEW vs STREAMING TABLE**
A `MATERIALIZED VIEW` is recomputed from scratch on each pipeline run — Lakeflow re-reads the entire source and replaces the result. This is correct for aggregations (`SUM`, `COUNT`, `GROUP BY`) where appending would produce wrong totals.

A `STREAMING TABLE` at Gold (like `fact_sales`) processes records incrementally — each batch from Silver adds new rows. This works for fact tables where records are immutable.

**Rule of thumb:** `MATERIALIZED VIEW` for dimension tables and aggregated summaries. `STREAMING TABLE` for fact tables and append-only datasets.

In [ ]:
# TODO: Complete the Gold declaration
# Gold reads silver_orders as a BATCH source (no STREAM) and aggregates it
# Use CREATE OR REFRESH MATERIALIZED VIEW (not STREAMING TABLE) for aggregate tables

gold_sql = """
-- TODO: Write the full CREATE OR REFRESH ... TABLE gold_daily_revenue declaration
-- Include the AS SELECT aggregation below
AS SELECT 
    order_date,
    SUM(total_price) AS total_revenue,
    COUNT(*) AS total_orders,
    AVG(total_price) AS avg_order_value
FROM silver_orders
GROUP BY order_date
ORDER BY order_date;
"""

print("Gold SQL declaration:")
print(gold_sql)

In [ ]:
# -- Validation --
assert "MATERIALIZED VIEW" in gold_sql.upper(), "Gold should use MATERIALIZED VIEW"
assert "silver" in gold_sql.lower(), "Should reference silver_orders"
print("Task 3 OK: Gold Materialized View declaration correct")

## Task 4: Classify STREAMING TABLE vs MATERIALIZED VIEW

Your pipeline (Section 1) created both object types. Prove you can tell them apart **from catalog metadata**, not just from the source SQL.

> Requires the Section 1 pipeline to have completed at least one run.

| Feature | STREAMING TABLE | MATERIALIZED VIEW |
|---------|----------------|-------------------|
| Processing mode | Incremental (append-only) | Full recomputation (or incremental where supported) |
| Best for | Append-only sources (files, CDC) | Aggregations, joins, dimension tables |
| Read from source | `STREAM(table_name)` | `table_name` |
| Supports expectations | Yes | Yes |

**What you need to do:**
1. Fill in the `answer` dict — classify `fact_sales` and `dim_customer`
2. Run the metadata query (provided) and make the validation pass

**Guidance — Task 04**

The goal is to solidify the **difference between STREAMING TABLE and MATERIALIZED VIEW** — a core exam concept — and to check it against the catalog.

**Two ways to inspect the object type**
1. `information_schema` (used in this task):
```sql
SELECT table_name, table_type
FROM <catalog>.information_schema.tables
WHERE table_schema = '<pipeline_target_schema>'
```
`table_type` returns `STREAMING_TABLE` or `MATERIALIZED_VIEW` for pipeline objects.

2. `DESCRIBE EXTENDED <table>` — look at the `Type` row.

**How to reason about it**
- `fact_sales` reads `FROM STREAM(silver_orders)` → incremental appends → **STREAMING_TABLE**
- `dim_customer` is declared `CREATE OR REFRESH MATERIALIZED VIEW` → recomputed per refresh → **MATERIALIZED_VIEW**

**Exam Tip:** If a query uses `FROM STREAM(table)`, the target **must** be a STREAMING TABLE. A MATERIALIZED VIEW cannot reference `STREAM(...)`.

In [ ]:
# TODO: Classify the two pipeline objects.
# Allowed values: "STREAMING_TABLE" or "MATERIALIZED_VIEW"
answer = {
    "fact_sales":   "...",   # reads FROM STREAM(silver_orders)
    "dim_customer": "...",   # declared CREATE OR REFRESH MATERIALIZED VIEW
}

# Provided: fetch the actual types from information_schema
type_df = spark.sql(f"""
    SELECT table_name, table_type
    FROM {CATALOG}.information_schema.tables
    WHERE table_schema = '{user_schema}'
      AND table_name IN ('fact_sales', 'dim_customer')
""")
actual = {r["table_name"]: r["table_type"] for r in type_df.collect()}
display(type_df)

In [ ]:
# -- Validation --
assert len(actual) == 2, (
    f"Expected fact_sales and dim_customer in {CATALOG}.{user_schema} — "
    "run the Section 1 pipeline first"
)
assert answer["fact_sales"] == "STREAMING_TABLE", \
    "fact_sales reads FROM STREAM(...) -> it must be a STREAMING_TABLE"
assert answer["dim_customer"] == "MATERIALIZED_VIEW", \
    "dim_customer is CREATE OR REFRESH MATERIALIZED VIEW"
assert answer == actual, f"Catalog disagrees with your answer: {actual}"
print("Task 4 OK: catalog metadata confirms ST vs MV classification")

## Task 5: Verify Pipeline Results (run after your pipeline completes)

Query the medallion layers your pipeline produced and verify the row-count relationships hold.

> Requires the Section 1 pipeline to have completed at least one run.

**What you need to do:** Fill in the three `spark.table(...)` counts for `bronze_orders`, `silver_orders`, and `fact_sales` in the pipeline target schema.

**Guidance — Task 05**

**Where pipeline tables live**
Everything the pipeline declares lands in the Target Schema you configured: `{CATALOG}.{user_schema}.<table>`. They are queryable like any table: `spark.table(f"{CATALOG}.{user_schema}.bronze_orders")`.

**What must hold after a healthy run**
- `bronze_orders` ≥ 100,000 (the batch backfill FLOW) plus any stream files
- `silver_orders` **≤** `bronze_orders` — the five `ON VIOLATION DROP ROW` expectations remove dirty rows
- `fact_sales` == `silver_orders` — each silver order becomes exactly one fact row (left joins to dims never drop rows)

**Bonus check:** the `is_unknown_customer` flag in `fact_sales` marks orders whose `customer_key` fell back to `-1`.

In [ ]:
# TODO: count rows in each layer of YOUR pipeline schema
# Tables: bronze_orders, silver_orders, fact_sales  (all in {CATALOG}.{user_schema})
bronze_cnt = ...  # YOUR CODE HERE
silver_cnt = ...  # YOUR CODE HERE
fact_cnt   = ...  # YOUR CODE HERE

print(f"bronze_orders : {bronze_cnt:,}")
print(f"silver_orders : {silver_cnt:,}")
print(f"fact_sales    : {fact_cnt:,}")

In [ ]:
# -- Validation --
assert bronze_cnt >= 100_000, \
    f"bronze_orders should hold at least the 100k batch backfill, got {bronze_cnt:,}"
assert 0 < silver_cnt <= bronze_cnt, \
    f"silver must be non-empty and <= bronze (expectations DROP ROW): {silver_cnt:,} vs {bronze_cnt:,}"
assert fact_cnt == silver_cnt, \
    f"fact_sales should have one row per silver order: {fact_cnt:,} != {silver_cnt:,}"
print(f"Task 5 OK: bronze={bronze_cnt:,} -> silver={silver_cnt:,} -> fact={fact_cnt:,}")

## Task 6: Query the Pipeline Event Log (run after your pipeline completes)

> 🏃 **Stretch** — optional in class (6-hour day): do it if you finish early, or after the course.

Read the expectations metrics that Lakeflow recorded for every flow — the audit trail behind the Data Quality tab.

> Requires the Section 1 pipeline to have completed at least one run.

**What you need to do:**
1. Query `event_log(TABLE(...))` for `flow_progress` events of `silver_orders`
2. Parse the `expectations` JSON array and aggregate passed/failed records per constraint

**Guidance — Task 06**

**The `event_log` table function**
Every pipeline writes an event log. For Unity Catalog pipelines, you can access the slice for one table with:
```sql
SELECT * FROM event_log(TABLE(catalog.schema.silver_orders))
```

**JSON path navigation with `:`**
`details:flow_progress.data_quality.expectations` extracts the expectations array (as a JSON string) from the event payload. Filter to `event_type = 'flow_progress'` rows where it is not null.

**Parsing the array in PySpark**
```python
from pyspark.sql.functions import from_json, explode, col
exp_schema = "array<struct<name:string, dataset:string, passed_records:bigint, failed_records:bigint>>"
parsed = (raw
    .withColumn("exp", explode(from_json(col("expectations"), exp_schema)))
    .select("exp.name", "exp.passed_records", "exp.failed_records"))
```

**What to look for**
- `failed_records` > 0 on constraints like `valid_quantity` — those are the intentionally dirty rows
- `passed_records` in the millions? You re-ran the pipeline — the log accumulates per flow update, so aggregate with `SUM`.

In [ ]:
from pyspark.sql.functions import from_json, explode, col

# TODO 1: read expectations events for silver_orders from the event log
# SELECT details:flow_progress.data_quality.expectations AS expectations
# FROM event_log(TABLE({CATALOG}.{user_schema}.silver_orders))
# WHERE event_type = 'flow_progress'
#   AND details:flow_progress.data_quality.expectations IS NOT NULL
raw_events = spark.sql(f"""
    -- YOUR CODE HERE
""")

# TODO 2: parse + aggregate per constraint (schema provided)
exp_schema = "array<struct<name:string, dataset:string, passed_records:bigint, failed_records:bigint>>"
dq_df = (
    # YOUR CODE HERE —
    #   explode(from_json(col("expectations"), exp_schema)) as "exp",
    #   then groupBy exp.name and SUM passed_records / failed_records
    ...
)
display(dq_df)

In [ ]:
# -- Validation --
dq_rows = {r["name"]: r for r in dq_df.collect()}
assert len(dq_rows) >= 3, \
    f"Expected the silver_orders constraints (valid_order_id, valid_quantity, ...), got {list(dq_rows)}"
total_failed = sum(r["failed_records"] for r in dq_rows.values())
total_passed = sum(r["passed_records"] for r in dq_rows.values())
assert total_passed > 0, "Event log should show passed records"
assert total_failed > 0, \
    "The RetailHub feed contains ~3% dirty rows — expected failed_records > 0"
print(f"Task 6 OK: {len(dq_rows)} constraints tracked, "
      f"{total_passed:,} passed / {total_failed:,} failed records logged")

## Task 7: Data-Quality Quarantine — Where Did the Dirty Rows Go?

> 🏃 **Stretch** — optional in class (6-hour day): do it if you finish early, or after the course.

The RetailHub feed intentionally contains **~3% dirty rows** (null keys, zero quantities, missing prices). The Silver expectations use `ON VIOLATION DROP ROW` — those rows never reach Silver. Quantify the quarantine and prove Silver is clean.

> Requires the Section 1 pipeline to have completed at least one run.

**What you need to do:**
1. Compute `dropped_rows` = bronze count − silver count, and the drop percentage
2. Count rows still in **bronze** that violate any Silver constraint (`violations_in_bronze`)
3. Count rows in **silver** that violate any constraint (`violations_in_silver`) — must be **zero**

**Guidance — Task 07**

**The quarantine equation**
Every bronze row either passes all five expectations into Silver or is dropped:
`bronze_cnt - silver_cnt == rows dropped by expectations`

**Reproducing the constraint predicate in SQL**
The Silver declaration drops a row when ANY of these fail:
```sql
order_id IS NOT NULL
customer_id IS NOT NULL
product_id IS NOT NULL
quantity IS NOT NULL AND quantity <> 0
unit_price IS NOT NULL AND unit_price >= 0
```
So a *violating* row satisfies the negation:
```sql
order_id IS NULL OR customer_id IS NULL OR product_id IS NULL
OR quantity IS NULL OR quantity = 0
OR unit_price IS NULL OR unit_price < 0
```
Count that predicate in bronze (should match `dropped_rows`) and in silver (must be 0).

**Why this matters in production**
`DROP ROW` is silent — without this audit you would never notice a broken upstream feed. Production patterns route violations to a *quarantine table* instead of discarding them (a second flow with the inverted predicate).

In [ ]:
VIOLATION_PREDICATE = """
    order_id IS NULL OR customer_id IS NULL OR product_id IS NULL
    OR quantity IS NULL OR quantity = 0
    OR unit_price IS NULL OR unit_price < 0
"""

# TODO 1: dropped rows + percentage (reuse bronze_cnt / silver_cnt from Task 5)
dropped_rows = ...
dropped_pct  = ...

# TODO 2: count violating rows still visible in bronze_orders
violations_in_bronze = ...  # filter bronze_orders with VIOLATION_PREDICATE

# TODO 3: count violating rows in silver_orders (expect 0!)
violations_in_silver = ...

print(f"Dropped by expectations : {dropped_rows:,} rows ({dropped_pct:.2f}% of bronze)")
print(f"Violations in bronze    : {violations_in_bronze:,}")
print(f"Violations in silver    : {violations_in_silver:,}")

In [ ]:
# -- Validation --
assert dropped_rows > 0, "Expected dropped rows — the feed contains ~3% dirty data"
assert 0 < dropped_pct < 10, \
    f"Drop rate should be a few percent (~3%), got {dropped_pct:.2f}% — check your counts"
assert violations_in_bronze == dropped_rows, (
    f"Every dropped row should still be visible in bronze: "
    f"{violations_in_bronze:,} violations vs {dropped_rows:,} dropped"
)
assert violations_in_silver == 0, \
    f"Silver must be clean — found {violations_in_silver} constraint-violating rows"
print(f"Task 7 OK: {dropped_rows:,} dirty rows ({dropped_pct:.2f}%) quarantined out of Silver — Silver is clean")

## Lab Complete!

You have:
- Written Bronze STREAMING TABLE declarations
- Written Silver declarations with data quality expectations
- Written Gold MATERIALIZED VIEW declarations
- Classified ST vs MV from `information_schema` metadata
- Verified layer-by-layer row counts after the pipeline run
- Queried the Event Log (`event_log(TABLE(...))`) for expectations metrics
- Audited the quarantine: ~3% dirty rows dropped, Silver provably clean

> **Exam Tip:** In Spark Declarative Pipelines, tables within the same pipeline reference each other directly by name — no prefix needed. Use `STREAM(table_name)` for streaming reads and just `table_name` for batch reads. `ON VIOLATION DROP ROW` silently drops and logs — it does NOT fail the pipeline.

> **Next:** LAB 08 - Lakeflow Jobs & Orchestration

## Cleanup (Optional)

In [ ]:
# Pipeline cleanup is done via Lakeflow UI (delete the pipeline)
print("LAB 07 complete. Delete the pipeline from Lakeflow UI when done.")

← [07 — Lakeflow Pipelines](../demo/07_lakeflow_pipelines.ipynb) | **[ README](../../../README.md)** | [08 — Job Orchestration →](../../day3/demo/08_job_orchestration.ipynb)